In [1]:
! pip install yfinance

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
from scipy.stats import norm, t
import matplotlib.pyplot as plt

In [3]:
stockList = ['MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOGL']
stocks = [stock for stock in stockList]

In [4]:
endDate = dt.datetime.now()
startDate = endDate - dt.timedelta(days=100)
stockData = yf.download(stocks, start=startDate, end=endDate)

/tmp/ipykernel_1091/2688441296.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stockData = yf.download(stocks, start=startDate, end=endDate)
[*********************100%***********************]  6 of 6 completed


In [5]:
stockData.tail(2)

Price            Close                                                 \
Ticker            AAPL        AMZN       GOOGL       META        MSFT   
Date                                                                    
2026-05-27  310.850006  271.850006  388.829987  635.26001  412.670013   
2026-05-28  310.339996  270.160004  388.890106  631.00000  424.329987   

Price                        High                                 ...  \
Ticker            NVDA       AAPL        AMZN       GOOGL   META  ...   
Date                                                              ...   
2026-05-27  212.600006  313.26001  272.410004  393.880005  638.5  ...   
2026-05-28  213.599197  312.76001  272.320007  391.500000  643.0  ...   

Price             Open                                        Volume  \
Ticker           GOOGL        META        MSFT        NVDA      AAPL   
Date                                                                   
2026-05-27  386.670013  609.400024  411.010010  214.119995  50388000   
2026-05-28  387.954987  639.979980  412.975006  211.274994  16703869   

Price                                                          
Ticker          AMZN     GOOGL      META      MSFT       NVDA  
Date                                                           
2026-05-27  39951900  23055700  23006900  28824000  167028500  
2026-05-28  17782686   8817539   9525326  23388419   68192912  

[2 rows x 30 columns]

In [6]:
stockData.swaplevel(axis=1).sort_index(axis=1).tail(2)

Ticker            AAPL                                               \
Price            Close       High         Low        Open    Volume   
Date                                                                  
2026-05-27  310.850006  313.26001  308.299988  308.329987  50388000   
2026-05-28  310.339996  312.76001  309.570007  310.679993  16703869   

Ticker            AMZN                                                ...  \
Price            Close        High         Low        Open    Volume  ...   
Date                                                                  ...   
2026-05-27  271.850006  272.410004  265.700012  266.149994  39951900  ...   
2026-05-28  270.160004  272.320007  267.440002  272.239990  17782686  ...   

Ticker            MSFT                                                \
Price            Close        High         Low        Open    Volume   
Date                                                                   
2026-05-27  412.670013  415.940002  409.579987  411.010010  28824000   
2026-05-28  424.329987  429.489990  412.670013  412.975006  23388419   

Ticker            NVDA                                                 
Price            Close        High         Low        Open     Volume  
Date                                                                   
2026-05-27  212.600006  214.149994  208.779999  214.119995  167028500  
2026-05-28  213.599197  214.289993  211.222000  211.274994   68192912  

[2 rows x 30 columns]

In [7]:
stockData['Close'].tail()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
Date,,,,,,
2026-05-21,304.989990,268.459991,387.660004,607.380005,419.089996,219.509995
2026-05-22,308.820007,266.320007,382.970001,610.260010,418.570007,215.330002
2026-05-26,308.329987,265.290009,388.880005,612.340027,416.029999,214.860001
2026-05-27,310.850006,271.850006,388.829987,635.260010,412.670013,212.600006
2026-05-28,310.339996,270.160004,388.890106,631.000000,424.329987,213.599197


$$\mathrm{Return}(t)=\frac{\mathrm{Price}(t)-\mathrm{Price}(t-1)}{\mathrm{Price}(t-1)}$$

In [8]:
returns = stockData['Close'].pct_change()
returns.tail()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
Date,,,,,,
2026-05-21,0.009065,0.013018,-0.003214,0.003834,-0.002523,-0.017721
2026-05-22,0.012558,-0.007971,-0.012098,0.004742,-0.001241,-0.019042
2026-05-26,-0.001587,-0.003868,0.015432,0.003408,-0.006068,-0.002183
2026-05-27,0.008173,0.024728,-0.000129,0.037430,-0.008076,-0.010518
2026-05-28,-0.001641,-0.006217,0.000155,-0.006706,0.028255,0.004700


In [ ]:
returns.shape

(71, 6)

La cantidad de registros corresponde a la cantidad de días quitando fines de semana y feriados.

In [ ]:
mean_returns = returns.mean()
mean_returns

,0
Ticker,
AAPL,0.002423
AMZN,0.004377
GOOGL,0.003842
META,0.000116
MSFT,0.001159
NVDA,0.002320


In [ ]:
cov_matrix = returns.cov()
cov_matrix

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA
Ticker,,,,,,
AAPL,0.000183,0.000076,0.000093,0.000104,0.000058,0.000098
AMZN,0.000076,0.000315,0.000196,0.000272,0.000093,0.000168
GOOGL,0.000093,0.000196,0.000444,0.000142,0.000037,0.000140
META,0.000104,0.000272,0.000142,0.000582,0.000228,0.000339
MSFT,0.000058,0.000093,0.000037,0.000228,0.000283,0.000147
NVDA,0.000098,0.000168,0.000140,0.000339,0.000147,0.000530


La diagonal es la varianza de cada activo respectivamente. En finanzas, la varianza mide qué tan volátil es una acción por sí sola.

In [12]:
returns = returns.dropna()

In [13]:
# Generamos pesos aleatorios para el portafolio que sumen 1
weights = np.random.random(len(returns.columns))
weights = weights / np.sum(weights)
weights

array([0.13873818, 0.15994623, 0.14829547, 0.24623877, 0.0579272 ,
       0.24885415])

In [18]:
# Calculamos el rendimiento histórico del portafolio consolidado
returns['Portfolio'] = returns.dot(weights)
returns['Portfolio'].tail()

,Portfolio
Date,
2026-05-21,-0.000749
2026-05-22,-0.004970
2026-05-26,0.001394
2026-05-27,0.011201
2026-05-28,-0.000044


In [19]:
returns.tail()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,Portfolio
Date,,,,,,,
2026-05-21,0.009065,0.013018,-0.003214,0.003834,-0.002523,-0.017721,-0.000749
2026-05-22,0.012558,-0.007971,-0.012098,0.004742,-0.001241,-0.019042,-0.004970
2026-05-26,-0.001587,-0.003868,0.015432,0.003408,-0.006068,-0.002183,0.001394
2026-05-27,0.008173,0.024728,-0.000129,0.037430,-0.008076,-0.010518,0.011201
2026-05-28,-0.001641,-0.006217,0.000155,-0.006706,0.028255,0.004700,-0.000044


En la Teoría Moderna de Portafolios (de Harry Markowitz) el objetivo es proyectar en el futuro dos cosas fundamentales de un portafolio de inversión: cuánto dinero vas a ganar (Retorno) y cuánto riesgo estás asumiendo (Volatilidad/Desviación Estándar).

In [15]:
# Necesitamos los pesos de las acciones que compramos (weights), los rendimientos históricos promedio (mean_returns), 
# la matriz de covarianza (cov_matrix) y el horizonte de tiempo (time). 
# A partir de estos parámetros, calculamos el rendimiento esperado (expected_returns) y el riesgo del portafolio completo (std).

time = 100 
expected_returns = np.sum(mean_returns * weights) * time
std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(time)

In [16]:
print("Rendimiento esperado del portafolio: ", expected_returns)
print("Riesgo del portafolio (desviación estándar): ", std)

Rendimiento esperado del portafolio:  0.2278941983752875
Riesgo del portafolio (desviación estándar):  0.15226113179051226


Al multiplicar el rendimiento promedio pasado por Time, estamos asumiendo matemáticamente el supuesto de que el futuro se va a comportar exactamente igual que el promedio del pasado. Aunque rendimientos pasados no garantizan rendimientos futuros. Es un "punto de partida" estándar: En estadística, ante la total imposibilidad de adivinar el futuro, el promedio histórico es matemáticamente el estimador menos sesgado que tenemos a la mano.

Si queremos que no sea una simple proyección lineal del pasado, se usan técnicas más avanzadas para simular un horizonte de tiempo. Simulaciones de Monte Carlo: En lugar de multiplicar por un horizonte de tiempo, usamos el promedio histórico y la covarianza como "reglas de juego" para que la computadora simule por ejemplo 10,000 futuros posibles diferentes día por día, añadiendo aleatoriedad (ruido estadístico). Al final, no obtenemos un solo número, sino una distribución de probabilidades _(ej. "Hay un 70% de probabilidad de que tu portafolio rinda entre tal y tal valor")_.

In [30]:
# Parámetros globales de la simulación

initial_investment = 10000 # Inversión inicial en dólares
alpha = 5 # Significancia del 5% (equivalente a 95% de confianza)

El VaR (Value at Risk) es una métrica que dice cuál es la pérdida máxima esperada para un nivel de confianza determinado (un parámetro alpha=0.05 indica 95% de confianza). Si alpha=5, significa que estamos buscando el peor 5% de los días de la historia de nuestro portafolio. Es decir, que tenemos un 95% de confianza de que las pérdidas no superarán ese número.

In [24]:
# a: son los datos
# q: es el percentil que queremos calcular, en este caso, el percentil del 5% (alpha)
var = np.percentile(a = returns['Portfolio'], q = alpha)
print(f"El VaR del portafolio al {100-alpha}% de confianza es: {-var:.2%}")

El VaR del portafolio al 95% de confianza es: 2.00%


Por pura convención. El VaR es, por definición, una métrica de pérdida. Como la palabra "pérdida" ya implica que el dinero va hacia abajo, dejar el signo negativo sería una doble negación (sería como decir "tengo una pérdida de -2%"). Convertirlo a un porcentaje positivo limpia la lectura del análisis.

In [27]:
below_var = returns['Portfolio'] <= var
print(below_var)

Date
2026-02-18    False
2026-02-19    False
2026-02-20    False
2026-02-23    False
2026-02-24    False
              ...  
2026-05-21    False
2026-05-22    False
2026-05-26    False
2026-05-27    False
2026-05-28    False
Name: Portfolio, Length: 70, dtype: bool


El percentil 5 (alpha=5) corta esa fila exactamente en el punto donde se separa el 5% de los peores días del 95% de los días restantes.

El CVaR (Conditional Value at Risk también llamado Expected Shortfall) va un paso más allá del VaR. El VaR te dice dónde empieza la "zona de desastre", pero el CVaR te dice qué tan grave es el desastre una vez que cruzas la línea. Responde a: "Si tengo la mala suerte de caer en ese peor 5%, ¿cuál será la pérdida promedio?"

In [29]:
cvar = returns['Portfolio'][below_var].mean()
print(f"El CVaR del portafolio al {100-alpha}% de confianza es: {-cvar:.2%}")

El CVaR del portafolio al 95% de confianza es: 2.72%


el CVaR es el promedio de las peores pérdidas.

El VaR que calcula la función originalmente es diario (porque tus datos de rendimiento son diarios). Si quieres saber cuánta plata arriesgas a lo largo de un periodo más largo (por ejemplo, en un horizonte de Time = 100 días), multiplicas el VaR diario por la raíz cuadrada del tiempo, exactamente igual a como escalaste la volatilidad en la función anterior.

In [32]:
hVaR = var * np.sqrt(time)
hCVaR = cvar * np.sqrt(time)
print(f"El VaR histórico del portafolio al {100-alpha}% de confianza es: {-hVaR:.2%}")
print(f"El CVaR histórico del portafolio al {100-alpha}% de confianza es: {-hCVaR:.2%}")

El VaR histórico del portafolio al 95% de confianza es: 19.97%
El CVaR histórico del portafolio al 95% de confianza es: 27.16%


Interpretación:
 - "Tengo un 95% de confianza de que en los próximos 100 días no perderé más del 19% ($1900 USD) de mi dinero (si invierto $10_000 USD)".
 - "Sin embargo, si las cosas salen realmente mal y caigo en ese fatídico 5% de peores escenarios, la pérdida promedio que debo esperar es del 27% ($2700 USD)".

In [ ]:
# ==========================================
# 4. SIMULACIÓN DE MONTE CARLO
# ==========================================

mc_sims = 1000 # Incrementado a 1000 para mayor precisión matemática
T = time

meanM = np.full(shape=(T, len(weights)), fill_value=mean_returns).T
portfolio_sims = np.full(shape=(T, mc_sims), fill_value=0.0)

# Simulación de trayectorias con correlación de Cholesky
for m in range(0, mc_sims):
    Z = np.random.normal(size=(T, len(weights)))
    L = np.linalg.cholesky(cov_matrix)
    dailyReturns = meanM + np.inner(L, Z)
    portfolio_sims[:, m] = np.cumprod(np.inner(weights, dailyReturns.T) + 1) * initial_investment

# Funciones de riesgo para Monte Carlo
def mcVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        return np.percentile(returns, alpha)
    else:
        raise TypeError("Se esperaba una serie de Pandas.")

def mcCVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        belowVaR = returns <= mcVaR(returns, alpha=alpha)
        return returns[belowVaR].mean()
    else:
        raise TypeError("Se esperaba una serie de Pandas.")

# Extraemos los resultados del último día de la simulación
portResults = pd.Series(portfolio_sims[-1, :])

mc_VaR_val = initial_investment - mcVaR(portResults, alpha=alpha)
mc_CVaR_val = initial_investment - mcCVaR(portResults, alpha=alpha)

In [ ]:
# ==========================================
# 2. MÉTODO HISTÓRICO
# ==========================================

def historicalVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        return np.percentile(returns, alpha)
    elif isinstance(returns, pd.DataFrame):
        return returns.aggregate(historicalVaR, alpha=alpha)
    else:
        raise TypeError("Se esperaba un DataFrame o Series de Pandas")

def historicalCVaR(returns, alpha=5):
    if isinstance(returns, pd.Series):
        belowVaR = returns <= historicalVaR(returns, alpha=alpha)
        return returns[belowVaR].mean()
    elif isinstance(returns, pd.DataFrame):
        return returns.aggregate(historicalCVaR, alpha=alpha)
    else:
        raise TypeError("Se esperaba un DataFrame o Series de Pandas")

# Calculamos y escalamos los valores históricos
hVaR = -historicalVaR(returns['portfolio'], alpha=alpha) * np.sqrt(Time)
hCVaR = -historicalCVaR(returns['portfolio'], alpha=alpha) * np.sqrt(Time)

In [ ]:
# ==========================================
# 5. DESPLIEGUE DE RESULTADOS Y GRÁFICO
# ==========================================

print("-" * 50)
print(f"REPORTES DE RIESGO FINANCIERO (Inversión: ${initial_investment:,} a {time} días)")
print("-" * 50)
print(f"Rendimiento Esperado del Portafolio : ${round(initial_investment * pRet, 2)}")
print("\n--- Métricas de Value at Risk (VaR 95%) ---")
print(f" Histórico VaR                      : ${round(initial_investment * hVaR, 2)}")
print(f" Paramétrico Normal VaR             : ${round(initial_investment * normVaR, 2)}")
print(f" Paramétrico t-Student VaR          : ${round(initial_investment * tVaR, 2)}")
print(f" Monte Carlo VaR                    : ${round(mc_VaR_val, 2)}")

print("\n--- Métricas de Conditional VaR (CVaR 95%) ---")
print(f" Histórico CVaR                     : ${round(initial_investment * hCVaR, 2)}")
print(f" Paramétrico Normal CVaR            : ${round(initial_investment * normCVaR, 2)}")
print(f" Paramétrico t-Student CVaR         : ${round(initial_investment * tCVaR, 2)}")
print(f" Monte Carlo CVaR                   : ${round(mc_CVaR_val, 2)}")
print("-" * 50)

# Generación del gráfico de Monte Carlo
plt.figure(figsize=(10, 6))
plt.plot(portfolio_sims, lw=0.5, alpha=0.6)
plt.axhline(initial_investment, color='black', linestyle='--', label='Inversión Inicial')
plt.ylabel('Valor del Portafolio ($)')
plt.xlabel('Días en el Futuro')
plt.title(f'Simulación de Monte Carlo ({mc_sims} escenarios a {T} días)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ==========================================
# 1. OBTENCIÓN DE DATOS Y CONFIGURACIÓN
# ==========================================

def getData(stocks, start, end):
    # Descargamos los datos usando yfinance 
    stockData = yf.download(stocks, start=start, end=end)
    stockData = stockData['Close']
    
    # Calculamos rendimientos diarios logarítmicos/porcentuales
    returns = stockData.pct_change()
    meanReturns = returns.mean()
    covMatrix = returns.cov()
    return returns, meanReturns, covMatrix

def portfolioPerformance(weights, meanReturns, covMatrix, Time):
    returns = np.sum(meanReturns * weights) * Time
    std = np.sqrt(np.dot(weights.T, np.dot(covMatrix, weights))) * np.sqrt(Time)
    return returns, std

# Definimos los activos más populares del S&P 500 (Mercado de EE. UU.)
stockList = ['MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOGL']
stocks = [stock for stock in stockList]

# Definimos la ventana de tiempo (800 días hacia atrás)
endDate = dt.datetime.now()
startDate = endDate - dt.timedelta(days=800)

# Descarga y limpieza
returns, meanReturns, covMatrix = getData(stocks, start=startDate, end=endDate)
returns = returns.dropna()

# Generamos pesos aleatorios para el portafolio que sumen 1
weights = np.random.random(len(returns.columns))
weights /= np.sum(weights)

# Calculamos el rendimiento histórico del portafolio consolidado
returns['portfolio'] = returns.dot(weights)

# Parámetros globales de la simulación
Time = 100 # Horizonte de tiempo en días
InitialInvestment = 10000 # Inversión inicial en dólares
alpha = 5 # Significancia del 5% (equivalente a 95% de confianza)

# Rendimiento y desviación estándar esperados del portafolio en el horizonte de tiempo
pRet, pStd = portfolioPerformance(weights, meanReturns, covMatrix, Time)

In [ ]:
# ==========================================
# 3. MÉTODO PARAMÉTRICO (Varianza-Covarianza)
# ==========================================

def var_parametric(portfolioReturns, portfolioStd, distribution='normal', alpha=5, dof=6):
    if distribution == 'normal':
        VaR = norm.ppf(1 - alpha/100) * portfolioStd - portfolioReturns
    elif distribution == 't-distribution':
        nu = dof
        VaR = np.sqrt((nu - 2) / nu) * t.ppf(1 - alpha/100, nu) * portfolioStd - portfolioReturns
    else:
        raise TypeError("Distribución no soportada. Use 'normal' o 't-distribution'")
    return VaR

def cvar_parametric(portfolioReturns, portfolioStd, distribution='normal', alpha=5, dof=6):
    if distribution == 'normal':
        CVaR = (alpha/100)**-1 * norm.pdf(norm.ppf(alpha/100)) * portfolioStd - portfolioReturns
    elif distribution == 't-distribution':
        nu = dof
        xanu = t.ppf(alpha/100, nu)
        CVaR = -1/(alpha/100) * (1 - nu)**(-1) * (nu - 2 + xanu**2) * t.pdf(xanu, nu) * portfolioStd - portfolioReturns
    else:
        raise TypeError("Distribución no soportada. Use 'normal' o 't-distribution'")
    return CVaR

# Cálculos Paramétricos
normVaR = var_parametric(pRet, pStd, distribution='normal', alpha=alpha)
normCVaR = cvar_parametric(pRet, pStd, distribution='normal', alpha=alpha)

tVaR = var_parametric(pRet, pStd, distribution='t-distribution', alpha=alpha, dof=6)
tCVaR = cvar_parametric(pRet, pStd, distribution='t-distribution', alpha=alpha, dof=6)